In [1]:
#%pip install lightgbm xgboost catboost


In [ ]:
import os
import pandas as pd
import joblib
from datetime import datetime
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from src.config import *
from src.utils import get_latest_file
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline


start_time = datetime.now()


# Charger les données
dataset_path = get_latest_file(DATA_FINAL_CLEANED_DATASET_DIR)
df = pd.read_csv(dataset_path)

# Préparation des données
target = 'IS_WIN'
drop_cols = COLS_TO_DROP_TARGET_IS_WIN + COLS_ODDS
features = [col for col in df.columns if col not in drop_cols + [target]]
df = df.dropna(subset=features)

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Définis tes modèles de base
estimators = [
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ('lgbm', LGBMClassifier(n_estimators=150, num_leaves=64, random_state=42, n_jobs=-1)),
    ('xgb', XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, subsample=0.7, colsample_bytree=0.7, random_state=42, eval_metric='logloss', n_jobs=-1, use_label_encoder=False)),
    ('cat', CatBoostClassifier(n_estimators=200, learning_rate=0.05, depth=6, rsm=0.8, verbose=0, random_state=42)),
    ('hgb', HistGradientBoostingClassifier(max_iter=200, random_state=42)),
    ('lr', make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, solver='liblinear', penalty='l2'))),
    ('mlp', make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42))),
    ('et', ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ('knn', make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15, n_jobs=-1)))
]

# Meta-model (peut être LogisticRegression, simple et efficace)
meta_model = LogisticRegression(solver='lbfgs', max_iter=5000)

# Création du stacking
stack = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_model,
    passthrough=False,
    cv=5,
    n_jobs=-1,
    verbose=2
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', stack)
])

# Entraînement
pipeline.fit(X_train, y_train)

# Évaluation
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
print("ROC AUC:", roc_auc_score(y_test, y_pred_proba))

# Sauvegarde
today = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

os.makedirs(DATA_MODELS_DIR, exist_ok=True)

stacking_model_path = os.path.join(DATA_MODELS_DIR, f"stacking_model_target_iswin_{today}.joblib")
joblib.dump(pipeline,stacking_model_path)
print(f"Model saved in {stacking_model_path}")

end_time = datetime.now()
print(f"Total execution time: {end_time - start_time}")



e:\Documents_\Dev\NBA_Predictor\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


ROC AUC: 0.731229063024896
Model saved in data\models\stacking_model_target_iswin_2025-06-05_17-47-58.joblib


In [ ]:
import os
import pandas as pd
import joblib
from datetime import datetime
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from src.config import *
from src.utils import get_latest_file
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier

from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

from sklearn.ensemble import RandomForestRegressor, StackingRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Charger les données
dataset_path = get_latest_file(DATA_FINAL_CLEANED_DATASET_DIR)
df = pd.read_csv(dataset_path)

# Préparation des données
target = 'POINT_DIFF'
drop_cols = COLS_TO_DROP_TARGET_POINT_DIFF
features = [col for col in df.columns if col not in drop_cols + [target]]
df = df.dropna(subset=features)

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Définis tes modèles de base
estimators = [
    ('rf', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ('lgbm', LGBMRegressor(n_estimators=150, num_leaves=64, random_state=42, n_jobs=-1)),
    ('xgb', XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, subsample=0.7, colsample_bytree=0.7, random_state=42, n_jobs=-1)),
    ('cat', CatBoostRegressor(n_estimators=200, learning_rate=0.05, depth=6, rsm=0.8, verbose=0, random_state=42)),
    ('hgb', HistGradientBoostingRegressor(max_iter=200, random_state=42)),
    ('lr', make_pipeline(StandardScaler(), LinearRegression())),
    ('mlp', make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42))),
    ('et', ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ('knn', make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=15, n_jobs=-1)))
]

# estimators_gpu = [
#     ('rf', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),  # CPU only
#     ('lgbm', LGBMRegressor(n_estimators=150, num_leaves=64, device='gpu', random_state=42, n_jobs=-1)),  # GPU
#     ('xgb', XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6,
#                          subsample=0.7, colsample_bytree=0.7, tree_method='gpu_hist',
#                          random_state=42, n_jobs=-1)),  # GPU
#     ('cat', CatBoostRegressor(n_estimators=200, learning_rate=0.05, depth=6,
#                               verbose=0, random_state=42, task_type='GPU', devices='0')),  # GPU
#     ('hgb', HistGradientBoostingRegressor(max_iter=200, random_state=42)),  # CPU only
#     ('lr', make_pipeline(StandardScaler(), LinearRegression())),  # CPU
#     ('mlp', make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42))),  # CPU
#     ('et', ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=-1)),  # CPU
#     ('knn', make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=15, n_jobs=-1)))  # CPU
# ]


meta_model = LinearRegression()

stack = StackingRegressor(
    estimators=estimators,
    final_estimator=meta_model,
    passthrough=False,
    cv=5,
    n_jobs=-1,
    verbose=2
)


pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', stack)
])

# Entraînement
pipeline.fit(X_train, y_train)

# Évaluation
y_pred = pipeline.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

# Sauvegarde
today = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

os.makedirs(DATA_MODELS_DIR, exist_ok=True)

stacking_model_path = os.path.join(DATA_MODELS_DIR, f"stacking_model_target_pointdiff_{today}.joblib")
joblib.dump(pipeline,stacking_model_path)
print(f"Model saved in {stacking_model_path}")


e:\Documents_\Dev\NBA_Predictor\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


MSE: 146.8474335851587
R² Score: 0.2325781907822303
Model saved in data\models\stacking_model_target_pointdiff_2025-06-05_03-26-42.joblib
